# Module 5: Control Flow for Bioinformatics

---

### Learning Objectives

By the end of this module, you will be able to:
- Use `if/elif/else` to classify biological data
- Apply `for` and `while` loops to iterate over sequences
- Control loop execution with `break`, `continue`, and `else`
- Use `enumerate()`, `zip()`, and `range()` effectively
- Translate codons to amino acids using a dictionary
- Find open reading frames (ORFs) in DNA sequences

### Prerequisites
- Python strings and string methods (Module 4)
- Basic understanding of DNA, RNA, codons, and amino acids

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
These patterns are used to build reusable data-processing scripts for FASTA/VCF/TSV handling, QC summaries, and automation glue code.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Mutable vs immutable objects and how that affects function behavior.
- Vectorized/dataframe workflows vs loops: both are useful, but for different bottlenecks.
- Reading errors: most failures come from dtype mismatches and unexpected missing values.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Module 4: Strings and Biological Sequences](../../Tier_1_Python_for_Bioinformatics/04_Strings_and_Sequences/01_strings_and_sequences.ipynb) | [Next: Module 6: Functions in Python →](../../Tier_1_Python_for_Bioinformatics/06_Functions/01_functions.ipynb)

---

## 1. Conditional Statements: if / elif / else

Conditionals let your program make decisions. In bioinformatics, we constantly classify, filter, and branch based on sequence properties.

```
         condition
           / \
        True  False
         |      |
      if block  else block
```

**Syntax:**
```python
if condition:
    # runs when condition is True
elif other_condition:
    # runs when other_condition is True
else:
    # runs when all conditions are False
```

In [ ]:
# Basic example: validate a nucleotide
nucleotide = "A"

if nucleotide in "ATGC":
    print(f"{nucleotide} is a valid DNA nucleotide")
elif nucleotide == "U":
    print(f"{nucleotide} is an RNA nucleotide (uracil)")
else:
    print(f"{nucleotide} is not a standard nucleotide")

### Classifying Nucleotides: Purines vs. Pyrimidines

- **Purines** (A, G): double-ring structure, larger molecules
- **Pyrimidines** (C, T, U): single-ring structure, smaller molecules

Chargaff's rule: in double-stranded DNA, the number of purines equals the number of pyrimidines.

In [ ]:
def classify_nucleotide(nuc):
    """Classify a nucleotide as purine or pyrimidine."""
    nuc = nuc.upper()
    if nuc in "AG":
        return "Purine (double-ring)"
    elif nuc in "CTU":
        return "Pyrimidine (single-ring)"
    else:
        return "Unknown"

for base in "ATGCU":
    print(f"{base}: {classify_nucleotide(base)}")

### GC Content Classification

GC content varies across genomes and has biological significance:

| GC Range | Classification | Examples |
|----------|---------------|----------|
| < 30% | AT-rich | Plasmodium falciparum (~20%) |
| 30-50% | Moderate | Homo sapiens (~41%) |
| 50-60% | High GC | Many bacterial genes |
| > 60% | Very high GC | Streptomyces (~72%) |

In [ ]:
def classify_gc_content(sequence):
    """Classify a DNA sequence by its GC content."""
    seq = sequence.upper()
    gc_count = seq.count('G') + seq.count('C')
    gc_percent = (gc_count / len(seq)) * 100
    
    if gc_percent < 30:
        category = "AT-rich"
    elif gc_percent < 50:
        category = "Moderate"
    elif gc_percent < 60:
        category = "High GC"
    else:
        category = "Very high GC"
    
    return gc_percent, category

# Test with sequences of different GC content
sequences = {
    "AT-rich region":   "AAATTTAAATTTAAATTTAAATTT",
    "Human average":    "ATGCGATCAATCGTATGCATGCA",
    "Bacterial gene":   "GCGATCGCGCGATCGCGATCGCG",
    "Streptomyces-like": "GCGCGCGCGCGCGCGCGCGCGCG"
}

print(f"{'Name':<20} {'Sequence':<26} {'GC%':>5}  Category")
print("-" * 70)
for name, seq in sequences.items():
    gc, cat = classify_gc_content(seq)
    print(f"{name:<20} {seq:<26} {gc:5.1f}%  {cat}")

### Detecting Sequence Type

Given an unknown biological sequence, we can determine whether it is DNA, RNA, or protein based on its character composition.

In [ ]:
def detect_sequence_type(sequence):
    """Detect whether a sequence is DNA, RNA, or protein."""
    seq = sequence.upper()
    unique_chars = set(seq)
    
    if unique_chars <= set("ATGC"):
        return "DNA"
    elif unique_chars <= set("AUGC"):
        return "RNA"
    elif unique_chars <= set("ACDEFGHIKLMNPQRSTVWY"):
        return "Protein"
    else:
        return "Unknown"

test_sequences = [
    "ATGCGATCGATCG",
    "AUGCGAUCGAUCG",
    "MKVLWAALLVLLGFANAT",
    "ATGXYZ123"
]

for seq in test_sequences:
    seq_type = detect_sequence_type(seq)
    print(f"{seq:<25} -> {seq_type}")

## 2. For Loops

For loops iterate over a sequence (string, list, range, etc.). They are the workhorse of bioinformatics programming.

```python
for item in iterable:
    # process item
```

In [ ]:
# Iterate over nucleotides in a sequence
dna = "ATGCGATC"

print("Counting nucleotides manually:")
a_count = 0
t_count = 0
g_count = 0
c_count = 0

for nucleotide in dna:
    if nucleotide == 'A':
        a_count += 1
    elif nucleotide == 'T':
        t_count += 1
    elif nucleotide == 'G':
        g_count += 1
    elif nucleotide == 'C':
        c_count += 1

print(f"A={a_count}, T={t_count}, G={g_count}, C={c_count}")

### The `range()` Function

Generates a sequence of integers. Essential for position-based iteration.

```
range(stop)              -> 0, 1, 2, ..., stop-1
range(start, stop)       -> start, start+1, ..., stop-1
range(start, stop, step) -> start, start+step, start+2*step, ...
```

In [ ]:
# range() examples
print("range(5):        ", list(range(5)))
print("range(1, 6):     ", list(range(1, 6)))
print("range(0, 15, 3): ", list(range(0, 15, 3)))  # codon start positions
print("range(10, 0, -1):", list(range(10, 0, -1)))  # countdown

In [ ]:
# Extract codons using range with step 3
dna = "ATGGCCGATCGATAGCCA"

print(f"DNA: {dna}")
print(f"Length: {len(dna)} bp")
print("\nCodons:")

codons = []
for i in range(0, len(dna) - 2, 3):
    codon = dna[i:i+3]
    if len(codon) == 3:  # only complete codons
        codons.append(codon)
        print(f"  Position {i:2d}-{i+2:2d}: {codon}")

print(f"\nTotal complete codons: {len(codons)}")

### `enumerate()`: Get Index and Value

When you need both the position and the value during iteration, use `enumerate()`.

In [ ]:
# Find positions of a specific nucleotide
dna = "ATGCGATCGATCGTAG"

print(f"Sequence: {dna}")
print(f"Positions of 'G':")

g_positions = []
for i, nuc in enumerate(dna):
    if nuc == 'G':
        g_positions.append(i)

print(f"  0-based: {g_positions}")
print(f"  1-based: {[p+1 for p in g_positions]}")

### `zip()`: Iterate Over Multiple Sequences in Parallel

`zip()` pairs up elements from two or more iterables. Very useful for comparing sequences, aligning positions, or combining related data.

In [ ]:
# Compare two aligned DNA sequences
seq1 = "ATGCGATCGA"
seq2 = "ATGCAATCTA"

print(f"Seq 1: {seq1}")
print(f"Seq 2: {seq2}")
print("Match: ", end="")

mismatches = 0
for nuc1, nuc2 in zip(seq1, seq2):
    if nuc1 == nuc2:
        print("|", end="")
    else:
        print("X", end="")
        mismatches += 1

print(f"\n\nMismatches: {mismatches}")
print(f"Identity: {(len(seq1) - mismatches) / len(seq1) * 100:.1f}%")

In [ ]:
# zip() with enumerate() for position-aware comparison
seq1 = "ATGCGATCGA"
seq2 = "ATGCAATCTA"

print("Mutation report:")
for i, (nuc1, nuc2) in enumerate(zip(seq1, seq2)):
    if nuc1 != nuc2:
        print(f"  Position {i+1}: {nuc1} -> {nuc2}")

In [ ]:
# zip() to pair gene names with their sequences
gene_names = ["BRCA1", "TP53", "EGFR"]
gene_lengths = [81189, 19149, 188307]
gc_contents = [41.8, 42.4, 48.2]

print(f"{'Gene':<10} {'Length':>10} {'GC%':>6}")
print("-" * 28)
for name, length, gc in zip(gene_names, gene_lengths, gc_contents):
    print(f"{name:<10} {length:>10,} {gc:>5.1f}%")

### Nested Loops

Loops inside loops. Useful for generating combinations, searching in 2D, or analyzing all reading frames.

In [ ]:
# Generate all 64 possible codons
nucleotides = "ATGC"
all_codons = []

for first in nucleotides:
    for second in nucleotides:
        for third in nucleotides:
            all_codons.append(first + second + third)

print(f"Total codons: {len(all_codons)}")
print(f"First 8:  {all_codons[:8]}")
print(f"Last 8:   {all_codons[-8:]}")

# How many start with 'A'?
a_start = [c for c in all_codons if c.startswith('A')]
print(f"\nCodons starting with A: {len(a_start)}")

## 3. While Loops

While loops keep running as long as a condition is True. Useful when you do not know in advance how many iterations you need.

```python
while condition:
    # loop body
    # must eventually make condition False!
```

In [ ]:
# Find the first stop codon in a reading frame
dna = "ATGGCCGATCGATAGCCATAGTTAACG"
stop_codons = {"TAA", "TAG", "TGA"}

position = 0
found = False

while position <= len(dna) - 3:
    codon = dna[position:position + 3]
    if codon in stop_codons:
        print(f"Stop codon '{codon}' found at position {position}")
        found = True
        break
    position += 3  # step by codon

if not found:
    print("No stop codon found in this reading frame")

In [ ]:
# Find all occurrences of a motif using while
dna = "ATGCGATGATCGATGCATG"
motif = "ATG"

positions = []
pos = dna.find(motif)
while pos != -1:
    positions.append(pos)
    pos = dna.find(motif, pos + 1)

print(f"Sequence: {dna}")
print(f"'{motif}' found at positions: {positions}")

## 4. Loop Control: break, continue, and the for-else Pattern

| Statement | Effect |
|-----------|--------|
| `break` | Exit the loop immediately |
| `continue` | Skip to the next iteration |
| `else` (on loop) | Runs only if the loop completed without `break` |

In [ ]:
# BREAK: stop at the first invalid character
sequence = "ATGCXGATC"
valid_bases = set("ATGC")

print(f"Validating: {sequence}")
for i, nuc in enumerate(sequence):
    if nuc not in valid_bases:
        print(f"Invalid character '{nuc}' at position {i}. Stopping.")
        break
else:
    # This block runs only if the loop did NOT break
    print("Sequence is valid!")

In [ ]:
# Validate a correct sequence -- the else clause runs
sequence = "ATGCGATC"
valid_bases = set("ATGC")

print(f"Validating: {sequence}")
for i, nuc in enumerate(sequence):
    if nuc not in valid_bases:
        print(f"Invalid character '{nuc}' at position {i}. Stopping.")
        break
else:
    print("Sequence is valid!")

In [ ]:
# CONTINUE: skip non-standard amino acids
protein = "MKVXLWABLLVZLLGFANAT"
standard_aa = set("ACDEFGHIKLMNPQRSTVWY")

clean_protein = []
skipped = 0

for aa in protein:
    if aa not in standard_aa:
        skipped += 1
        continue  # skip non-standard amino acids
    clean_protein.append(aa)

print(f"Original: {protein} ({len(protein)} aa)")
print(f"Cleaned:  {''.join(clean_protein)} ({len(clean_protein)} aa)")
print(f"Skipped {skipped} non-standard residues")

## 5. Codon Table: Translating DNA to Protein

The genetic code maps 64 codons to 20 amino acids (plus stop signals). We represent this as a Python dictionary. This is one of the most fundamental data structures in computational biology.

In [ ]:
# The Standard Genetic Code
CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

print(f"Total codons in table: {len(CODON_TABLE)}")
print(f"ATG encodes: {CODON_TABLE['ATG']} (Methionine, start codon)")
print(f"TAA encodes: {CODON_TABLE['TAA']} (stop codon)")
print(f"TGG encodes: {CODON_TABLE['TGG']} (Tryptophan, only 1 codon)")

In [ ]:
# Translate a DNA sequence to protein
def translate(dna):
    """Translate a DNA coding sequence into a protein sequence."""
    dna = dna.upper()
    protein = []
    
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i+3]
        if len(codon) < 3:
            break
        amino_acid = CODON_TABLE.get(codon, '?')
        if amino_acid == '*':  # stop codon
            break
        protein.append(amino_acid)
    
    return ''.join(protein)

# Test: translate a short gene
coding_seq = "ATGGCCGATCGTTAG"
protein = translate(coding_seq)

# Show codon-by-codon translation
print(f"DNA:     {coding_seq}")
codons = [coding_seq[i:i+3] for i in range(0, len(coding_seq), 3)]
aas = [CODON_TABLE.get(c, '?') for c in codons]
print(f"Codons:  {' '.join(codons)}")
print(f"AA:      {'   '.join(aas)}")
print(f"Protein: {protein}")

### Analyzing the Genetic Code

Let's use loops to explore properties of the genetic code itself.

In [ ]:
# How many codons encode each amino acid? (degeneracy of the genetic code)
aa_codons = {}  # amino acid -> list of codons

for codon, aa in CODON_TABLE.items():
    if aa not in aa_codons:
        aa_codons[aa] = []
    aa_codons[aa].append(codon)

print(f"{'AA':<5} {'Count':<6} {'Codons'}")
print("-" * 45)
for aa in sorted(aa_codons.keys()):
    codons = aa_codons[aa]
    label = "Stop" if aa == '*' else aa
    print(f"{label:<5} {len(codons):<6} {', '.join(codons)}")

## 6. Filtering Sequences by Length

A common task in bioinformatics pipelines: filter sequences based on some criterion (length, GC content, quality score, etc.).

In [ ]:
# Filter sequences by minimum length
sequences = {
    "read_001": "ATGCGATCGATCGTAGCATGCAT",
    "read_002": "ATGCGA",
    "read_003": "GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAG",
    "read_004": "ATG",
    "read_005": "ATGCGATCGATCGTAG",
    "read_006": "GCTAGCTAGCTAGCTAGCTAGCTA",
    "read_007": "AT",
}

min_length = 10
passed = []
failed = []

for name, seq in sequences.items():
    if len(seq) >= min_length:
        passed.append(name)
    else:
        failed.append(name)

print(f"Minimum length filter: {min_length} bp")
print(f"Passed: {len(passed)}/{len(sequences)} sequences")
print(f"Failed: {len(failed)}/{len(sequences)} sequences")

print(f"\nPassed reads:")
for name in passed:
    print(f"  {name}: {len(sequences[name])} bp")
print(f"\nFiltered out:")
for name in failed:
    print(f"  {name}: {len(sequences[name])} bp")

In [ ]:
# Filter by multiple criteria: length AND GC content
min_length = 10
min_gc = 40.0
max_gc = 65.0

print(f"Filters: length >= {min_length}, {min_gc}% <= GC <= {max_gc}%")
print(f"{'Read':<12} {'Length':>6} {'GC%':>6} {'Status'}")
print("-" * 40)

for name, seq in sequences.items():
    length = len(seq)
    gc = (seq.count('G') + seq.count('C')) / length * 100 if length > 0 else 0
    
    if length < min_length:
        status = "FAIL (too short)"
    elif gc < min_gc:
        status = "FAIL (low GC)"
    elif gc > max_gc:
        status = "FAIL (high GC)"
    else:
        status = "PASS"
    
    print(f"{name:<12} {length:>6} {gc:>5.1f}% {status}")

## 7. Finding Palindromic Sequences

In molecular biology, a **palindromic sequence** is one where the sequence on the sense strand reads the same as its reverse complement. Many restriction enzyme recognition sites are palindromic.

```
Example: EcoRI recognition site
5'-G A A T T C-3'
3'-C T T A A G-5'

Forward:  GAATTC
Rev comp: GAATTC  (same!)
```

This is different from a regular string palindrome (like "racecar")!

In [ ]:
def reverse_complement(dna):
    """Return the reverse complement of a DNA sequence."""
    comp_table = str.maketrans('ATGC', 'TACG')
    return dna.upper().translate(comp_table)[::-1]

def is_palindromic(dna):
    """Check if a DNA sequence is a biological palindrome (equals its reverse complement)."""
    dna = dna.upper()
    return dna == reverse_complement(dna)

# Test with known restriction enzyme sites
sites = {
    "EcoRI":  "GAATTC",
    "BamHI":  "GGATCC",
    "HindIII": "AAGCTT",
    "NotI":   "GCGGCCGC",
    "Random":  "ATGCGA",
}

print(f"{'Enzyme':<10} {'Site':<12} {'Rev Comp':<12} {'Palindrome?'}")
print("-" * 48)
for enzyme, site in sites.items():
    rc = reverse_complement(site)
    is_pal = is_palindromic(site)
    print(f"{enzyme:<10} {site:<12} {rc:<12} {is_pal}")

In [ ]:
# Find all palindromic subsequences of a given length in a sequence
def find_palindromes(sequence, min_len=4, max_len=8):
    """Find all palindromic subsequences in a DNA sequence."""
    sequence = sequence.upper()
    palindromes = []
    
    for length in range(min_len, max_len + 1, 2):  # palindromes must be even length
        for start in range(len(sequence) - length + 1):
            subseq = sequence[start:start + length]
            if is_palindromic(subseq):
                palindromes.append((start + 1, length, subseq))  # 1-based
    
    return palindromes

test_seq = "ATGAATTCGATGGATCCATGCAAGCTTGCA"
results = find_palindromes(test_seq)

print(f"Sequence: {test_seq}")
print(f"\nPalindromic subsequences (4-8 bp):")
print(f"{'Position':<10} {'Length':<8} {'Sequence'}")
print("-" * 30)
for pos, length, subseq in results:
    print(f"{pos:<10} {length:<8} {subseq}")

## 8. Sliding Window Analysis

A sliding window moves across a sequence, analyzing each window of fixed size. This is a fundamental technique for studying local sequence properties.

In [ ]:
# Sliding window GC content
def sliding_window_gc(sequence, window_size=10, step=1):
    """Calculate GC content in sliding windows across a DNA sequence."""
    sequence = sequence.upper()
    results = []
    
    for start in range(0, len(sequence) - window_size + 1, step):
        window = sequence[start:start + window_size]
        gc = (window.count('G') + window.count('C')) / window_size * 100
        results.append((start, gc))
    
    return results

# A sequence with distinct AT-rich and GC-rich regions
test_seq = "AAATTTAAATTTGCGCGCGCGCGCAAATTTAAATTT"

gc_profile = sliding_window_gc(test_seq, window_size=10, step=5)

print(f"Sequence: {test_seq}")
print(f"\nGC content profile (window=10bp, step=5bp):")
for pos, gc in gc_profile:
    bar = '#' * int(gc / 5)
    print(f"  Position {pos:3d}: {gc:5.1f}% {bar}")

## 9. Finding Open Reading Frames (ORFs)

An **Open Reading Frame (ORF)** is a stretch of DNA that begins with a start codon (ATG) and ends with a stop codon (TAA, TAG, or TGA), with no internal stop codons in the same reading frame.

```
Reading frame 1: ATG | GCC | GAT | TAG | ...
Reading frame 2:  TG | GCC | GAT | TAG | ...
Reading frame 3:   G | GCC | GAT | TAG | ...
```

A real gene-finding analysis checks all 3 forward frames and all 3 reverse complement frames (6 total).

In [ ]:
def find_orfs(sequence, min_length=0):
    """Find all ORFs in a DNA sequence across all 3 forward reading frames."""
    sequence = sequence.upper()
    stop_codons = {"TAA", "TAG", "TGA"}
    orfs = []
    
    # Check each reading frame
    for frame in range(3):
        i = frame
        while i <= len(sequence) - 3:
            codon = sequence[i:i+3]
            
            if codon == "ATG":
                # Found a start codon; look for the stop codon
                orf_start = i
                j = i + 3
                while j <= len(sequence) - 3:
                    next_codon = sequence[j:j+3]
                    if next_codon in stop_codons:
                        orf_end = j + 3
                        orf_seq = sequence[orf_start:orf_end]
                        if len(orf_seq) >= min_length:
                            orfs.append({
                                'frame': frame + 1,
                                'start': orf_start + 1,  # 1-based
                                'end': orf_end,
                                'length': orf_end - orf_start,
                                'sequence': orf_seq,
                                'protein': translate(orf_seq)
                            })
                        i = j + 3  # continue after stop codon
                        break
                    j += 3
                else:
                    # No stop codon found
                    i += 3
                    continue
                continue
            i += 3
    
    return orfs

# Test with a sequence containing multiple ORFs
dna = "AATGGCCGATTAGCCAATGAAATGACCCATGTAG"
orfs = find_orfs(dna)

print(f"Sequence: {dna}")
print(f"Length: {len(dna)} bp")
print(f"\nFound {len(orfs)} ORF(s):")
for orf in orfs:
    print(f"\n  Frame {orf['frame']}: position {orf['start']}-{orf['end']} ({orf['length']} bp)")
    print(f"  DNA:     {orf['sequence']}")
    print(f"  Protein: {orf['protein']}")

### ORF Analysis on Both Strands

A complete ORF finder checks 6 reading frames: 3 on the forward strand and 3 on the reverse complement.

In [ ]:
def find_all_orfs(sequence, min_length=30):
    """Find ORFs on both strands of a DNA sequence."""
    sequence = sequence.upper()
    
    # Forward strand
    forward_orfs = find_orfs(sequence, min_length)
    for orf in forward_orfs:
        orf['strand'] = '+'
    
    # Reverse complement strand
    rc = reverse_complement(sequence)
    reverse_orfs = find_orfs(rc, min_length)
    for orf in reverse_orfs:
        orf['strand'] = '-'
    
    return forward_orfs + reverse_orfs

# Test
dna = "AATGGCCGATTAGCCAATGAAATGACCCATGTAGCCGATCGATCGTAG"
all_orfs = find_all_orfs(dna, min_length=9)

print(f"Sequence ({len(dna)} bp):")
print(f"5'-{dna}-3'")
print(f"\nORFs found (min. 9 bp):")
print(f"{'Strand':<8} {'Frame':<7} {'Start':<7} {'End':<7} {'Length':<8} {'Protein'}")
print("-" * 55)
for orf in all_orfs:
    print(f"{orf['strand']:<8} {orf['frame']:<7} {orf['start']:<7} {orf['end']:<7} {orf['length']:<8} {orf['protein']}")

---

## Exercises

### Exercise 1: Codon Table Lookup (difficulty: *)

Write code that takes a DNA codon (3-letter string), looks it up in the codon table, and prints the amino acid with its properties. Use the CODON_TABLE defined earlier.

If the codon is invalid, print an error message.

In [ ]:
# Amino acid name lookup
AA_NAMES = {
    'A': 'Alanine', 'R': 'Arginine', 'N': 'Asparagine', 'D': 'Aspartate',
    'C': 'Cysteine', 'E': 'Glutamate', 'Q': 'Glutamine', 'G': 'Glycine',
    'H': 'Histidine', 'I': 'Isoleucine', 'L': 'Leucine', 'K': 'Lysine',
    'M': 'Methionine', 'F': 'Phenylalanine', 'P': 'Proline', 'S': 'Serine',
    'T': 'Threonine', 'W': 'Tryptophan', 'Y': 'Tyrosine', 'V': 'Valine',
    '*': 'Stop'
}

def lookup_codon(codon):
    """Look up a codon and print the amino acid with its full name."""
    codon = codon.upper()
    
    # YOUR CODE HERE
    # 1. Check if codon is exactly 3 characters
    # 2. Check if codon is in CODON_TABLE
    # 3. Print the amino acid letter and full name
    pass

# Test
for test_codon in ["ATG", "TAA", "GCT", "XYZ", "AT"]:
    lookup_codon(test_codon)
    print()

In [ ]:
# --- SOLUTION ---
def lookup_codon(codon):
    """Look up a codon and print the amino acid with its full name."""
    codon = codon.upper()
    
    if len(codon) != 3:
        print(f"Error: '{codon}' is not a valid codon (must be 3 characters)")
        return
    
    if codon not in CODON_TABLE:
        print(f"Error: '{codon}' is not a valid DNA codon")
        return
    
    aa = CODON_TABLE[codon]
    name = AA_NAMES.get(aa, "Unknown")
    
    if aa == '*':
        print(f"Codon {codon} -> Stop codon")
    else:
        print(f"Codon {codon} -> {aa} ({name})")

# Test
for test_codon in ["ATG", "TAA", "GCT", "XYZ", "AT"]:
    lookup_codon(test_codon)
    print()

### Exercise 2: Classify Amino Acids (difficulty: **)

Write a function that takes a protein sequence and classifies each amino acid by its chemical property. Return a summary of counts for each category.

Categories:
- **Nonpolar (hydrophobic):** G, A, V, L, I, P, F, M, W
- **Polar (uncharged):** S, T, C, Y, N, Q
- **Positively charged (+):** K, R, H
- **Negatively charged (-):** D, E

In [ ]:
def classify_protein(protein_seq):
    """Classify each amino acid in a protein and return a summary."""
    protein_seq = protein_seq.upper()
    
    # Define categories
    categories = {
        'Nonpolar': set('GAVILPFMW'),
        'Polar': set('STCYNQ'),
        'Positive': set('KRH'),
        'Negative': set('DE')
    }
    
    counts = {cat: 0 for cat in categories}
    
    # YOUR CODE HERE
    # Loop through each amino acid and classify it
    
    return counts

# Test with insulin B chain
insulin_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"
result = classify_protein(insulin_b)
print(f"Protein: {insulin_b}")
print(f"Length:  {len(insulin_b)} aa")
print(f"\nAmino acid classification:")
for cat, count in result.items():
    pct = count / len(insulin_b) * 100
    print(f"  {cat:<12}: {count:3d} ({pct:5.1f}%)")

In [ ]:
# --- SOLUTION ---
def classify_protein(protein_seq):
    """Classify each amino acid in a protein and return a summary."""
    protein_seq = protein_seq.upper()
    
    categories = {
        'Nonpolar': set('GAVILPFMW'),
        'Polar': set('STCYNQ'),
        'Positive': set('KRH'),
        'Negative': set('DE')
    }
    
    counts = {cat: 0 for cat in categories}
    
    for aa in protein_seq:
        for cat, aa_set in categories.items():
            if aa in aa_set:
                counts[cat] += 1
                break
    
    return counts

# Test with insulin B chain
insulin_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"
result = classify_protein(insulin_b)
print(f"Protein: {insulin_b}")
print(f"Length:  {len(insulin_b)} aa")
print(f"\nAmino acid classification:")
total_classified = sum(result.values())
for cat, count in result.items():
    pct = count / len(insulin_b) * 100
    bar = '#' * int(pct / 3)
    print(f"  {cat:<12}: {count:3d} ({pct:5.1f}%) {bar}")

# Overall: is this protein more hydrophobic or hydrophilic?
hydrophobic_pct = result['Nonpolar'] / len(insulin_b) * 100
print(f"\nHydrophobic content: {hydrophobic_pct:.1f}%")
if hydrophobic_pct > 50:
    print("This protein has a strong hydrophobic character.")
else:
    print("This protein has a mixed hydrophobic/hydrophilic character.")

### Exercise 3: Translate All Three Reading Frames (difficulty: **)

Write a function that translates a DNA sequence in all 3 forward reading frames and returns the results. For each frame, translate the entire sequence (not just from ATG).

Test with: `"ATGGCCGATCGATAGCCA"`

In [ ]:
def translate_all_frames(dna):
    """Translate a DNA sequence in all 3 forward reading frames."""
    dna = dna.upper()
    results = {}
    
    # YOUR CODE HERE
    # For frames 1, 2, 3:
    #   - Extract codons starting at offset 0, 1, 2
    #   - Look up each codon in CODON_TABLE
    #   - Store the protein string in results
    
    return results

# Test
test = "ATGGCCGATCGATAGCCA"
frames = translate_all_frames(test)
print(f"DNA: {test}")
for frame, protein in frames.items():
    print(f"Frame {frame}: {protein}")

In [ ]:
# --- SOLUTION ---
def translate_all_frames(dna):
    """Translate a DNA sequence in all 3 forward reading frames."""
    dna = dna.upper()
    results = {}
    
    for frame in range(3):
        protein = []
        for i in range(frame, len(dna) - 2, 3):
            codon = dna[i:i+3]
            if len(codon) == 3:
                aa = CODON_TABLE.get(codon, '?')
                protein.append(aa)
        results[frame + 1] = ''.join(protein)
    
    return results

# Test
test = "ATGGCCGATCGATAGCCA"
frames = translate_all_frames(test)
print(f"DNA: {test}")
print()
for frame, protein in frames.items():
    codons = [test[i:i+3] for i in range(frame-1, len(test)-2, 3) if i+3 <= len(test)]
    print(f"Frame {frame}: {' '.join(codons)}")
    print(f"         {'   '.join(protein)}")
    print()

### Exercise 4: Sequence Validator with Detailed Report (difficulty: **)

Write a function that validates a DNA sequence and produces a detailed report. Check:
1. Contains only valid characters (A, T, G, C)
2. Length is a multiple of 3
3. Starts with ATG
4. Ends with a stop codon
5. Has no internal stop codons

In [ ]:
def validate_cds(dna):
    """Validate a coding DNA sequence and return a report."""
    dna = dna.upper().strip()
    checks = []
    
    # YOUR CODE HERE
    # Perform each check and append (check_name, passed_bool, message) to checks
    
    return checks

# Test with various sequences
test_sequences = [
    "ATGGCCGATTAG",       # valid CDS
    "ATGGCCXATTAG",       # invalid character
    "ATGGCCGATCGA",       # no stop codon
    "GCCGATCGATAG",       # no start codon
    "ATGTAGGCCTAG",       # internal stop codon
]

for seq in test_sequences:
    print(f"\nSequence: {seq}")
    results = validate_cds(seq)
    for name, passed, msg in results:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}: {msg}")

In [ ]:
# --- SOLUTION ---
def validate_cds(dna):
    """Validate a coding DNA sequence and return a report."""
    dna = dna.upper().strip()
    checks = []
    stop_codons = {'TAA', 'TAG', 'TGA'}
    
    # Check 1: valid characters
    invalid_chars = set(dna) - set('ATGC')
    if invalid_chars:
        checks.append(("Valid characters", False, f"Invalid: {invalid_chars}"))
    else:
        checks.append(("Valid characters", True, "All characters are A, T, G, or C"))
    
    # Check 2: length multiple of 3
    if len(dna) % 3 == 0:
        checks.append(("Length", True, f"{len(dna)} bp (multiple of 3)"))
    else:
        checks.append(("Length", False, f"{len(dna)} bp (not a multiple of 3)"))
    
    # Check 3: starts with ATG
    if dna.startswith('ATG'):
        checks.append(("Start codon", True, "Starts with ATG"))
    else:
        checks.append(("Start codon", False, f"Starts with {dna[:3]}"))
    
    # Check 4: ends with stop codon
    last_codon = dna[-3:] if len(dna) >= 3 else ''
    if last_codon in stop_codons:
        checks.append(("Stop codon", True, f"Ends with {last_codon}"))
    else:
        checks.append(("Stop codon", False, f"Ends with {last_codon} (not a stop codon)"))
    
    # Check 5: no internal stop codons
    internal_stops = []
    for i in range(3, len(dna) - 3, 3):  # skip first and last codons
        codon = dna[i:i+3]
        if codon in stop_codons:
            internal_stops.append((i, codon))
    
    if not internal_stops:
        checks.append(("No internal stops", True, "No premature stop codons"))
    else:
        positions_str = ', '.join(f"{c} at pos {p}" for p, c in internal_stops)
        checks.append(("No internal stops", False, f"Found: {positions_str}"))
    
    return checks

# Test
test_sequences = [
    "ATGGCCGATTAG",
    "ATGGCCXATTAG",
    "ATGGCCGATCGA",
    "GCCGATCGATAG",
    "ATGTAGGCCTAG",
]

for seq in test_sequences:
    print(f"\nSequence: {seq}")
    results = validate_cds(seq)
    all_passed = all(passed for _, passed, _ in results)
    for name, passed, msg in results:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}: {msg}")
    print(f"  Overall: {'Valid CDS' if all_passed else 'Invalid CDS'}")

### Exercise 5: Find All ORFs in a Sequence (difficulty: ***)

Write your own ORF finder that searches all 6 reading frames (3 forward + 3 reverse complement). For each ORF, report the strand, frame, start/end positions, DNA sequence, and protein translation.

Use a minimum ORF length of 30 bp (10 amino acids including stop).

Test with:
```
AATGGCCGATCGATAGCCCATGAAAGCCGCTGATTAGCCAATGCGATCGATCGTAG
```

In [ ]:
def find_all_orfs_v2(sequence, min_length=30):
    """Find all ORFs on both strands (6 frames)."""
    # YOUR CODE HERE
    # 1. Search 3 forward frames
    # 2. Compute reverse complement
    # 3. Search 3 reverse frames
    # 4. Return list of ORF dictionaries
    pass

# Test
test_dna = "AATGGCCGATCGATAGCCCATGAAAGCCGCTGATTAGCCAATGCGATCGATCGTAG"
orfs = find_all_orfs_v2(test_dna, min_length=9)
if orfs:
    print(f"Found {len(orfs)} ORFs:")
    for orf in orfs:
        print(f"  Strand {orf['strand']}, Frame {orf['frame']}: "
              f"pos {orf['start']}-{orf['end']} ({orf['length']} bp) -> {orf['protein']}")

In [ ]:
# --- SOLUTION ---
def find_orfs_single_strand(sequence, min_length=30):
    """Find all ORFs on a single strand (3 frames)."""
    sequence = sequence.upper()
    stop_codons = {"TAA", "TAG", "TGA"}
    orfs = []
    
    for frame in range(3):
        i = frame
        while i <= len(sequence) - 3:
            codon = sequence[i:i+3]
            if codon == "ATG":
                orf_start = i
                j = i + 3
                while j <= len(sequence) - 3:
                    if sequence[j:j+3] in stop_codons:
                        orf_end = j + 3
                        orf_seq = sequence[orf_start:orf_end]
                        if len(orf_seq) >= min_length:
                            orfs.append({
                                'frame': frame + 1,
                                'start': orf_start + 1,
                                'end': orf_end,
                                'length': orf_end - orf_start,
                                'sequence': orf_seq,
                                'protein': translate(orf_seq)
                            })
                        i = j + 3
                        break
                    j += 3
                else:
                    i += 3
                    continue
                continue
            i += 3
    
    return orfs

def find_all_orfs_v2(sequence, min_length=30):
    """Find all ORFs on both strands (6 frames)."""
    sequence = sequence.upper()
    
    # Forward strand
    forward_orfs = find_orfs_single_strand(sequence, min_length)
    for orf in forward_orfs:
        orf['strand'] = '+'
    
    # Reverse complement strand
    rc = reverse_complement(sequence)
    reverse_orfs = find_orfs_single_strand(rc, min_length)
    for orf in reverse_orfs:
        orf['strand'] = '-'
    
    return forward_orfs + reverse_orfs

# Test
test_dna = "AATGGCCGATCGATAGCCCATGAAAGCCGCTGATTAGCCAATGCGATCGATCGTAG"
orfs = find_all_orfs_v2(test_dna, min_length=9)

print(f"Sequence ({len(test_dna)} bp):")
print(f"5'-{test_dna}-3'")
print(f"\nFound {len(orfs)} ORF(s):")
print(f"{'Strand':<8} {'Frame':<7} {'Start':<7} {'End':<7} {'Length':<8} {'DNA':<30} {'Protein'}")
print("-" * 85)
for orf in orfs:
    dna_display = orf['sequence'][:27] + '...' if len(orf['sequence']) > 30 else orf['sequence']
    print(f"{orf['strand']:<8} {orf['frame']:<7} {orf['start']:<7} {orf['end']:<7} {orf['length']:<8} {dna_display:<30} {orf['protein']}")

### Exercise 6: Hamming Distance and Mutation Counter (difficulty: **)

The **Hamming distance** between two strings of equal length is the number of positions at which they differ. Write a function that computes the Hamming distance and classifies each mutation as a **transition** (purine<->purine or pyrimidine<->pyrimidine) or **transversion** (purine<->pyrimidine).

- Transitions: A<->G, C<->T
- Transversions: A<->C, A<->T, G<->C, G<->T

In [ ]:
def mutation_analysis(seq1, seq2):
    """Compute Hamming distance and classify mutations."""
    # YOUR CODE HERE
    # 1. Check sequences are the same length
    # 2. Compare position by position using zip()
    # 3. Count transitions and transversions
    # 4. Return a dictionary with the results
    pass

# Test
s1 = "ATGCGATCGA"
s2 = "ATGCAATCTA"
result = mutation_analysis(s1, s2)
print(f"Seq 1: {s1}")
print(f"Seq 2: {s2}")
if result:
    print(f"Hamming distance:  {result['hamming']}")
    print(f"Transitions (Ti):  {result['transitions']}")
    print(f"Transversions (Tv): {result['transversions']}")
    print(f"Ti/Tv ratio: {result['ti_tv_ratio']:.2f}")

In [ ]:
# --- SOLUTION ---
def mutation_analysis(seq1, seq2):
    """Compute Hamming distance and classify mutations."""
    seq1 = seq1.upper()
    seq2 = seq2.upper()
    
    if len(seq1) != len(seq2):
        print("Error: sequences must be the same length")
        return None
    
    purines = set('AG')
    pyrimidines = set('CT')
    
    hamming = 0
    transitions = 0
    transversions = 0
    mutation_details = []
    
    for i, (n1, n2) in enumerate(zip(seq1, seq2)):
        if n1 != n2:
            hamming += 1
            # Both purines or both pyrimidines -> transition
            if (n1 in purines and n2 in purines) or (n1 in pyrimidines and n2 in pyrimidines):
                transitions += 1
                mutation_details.append((i + 1, n1, n2, "Ti"))
            else:
                transversions += 1
                mutation_details.append((i + 1, n1, n2, "Tv"))
    
    ti_tv_ratio = transitions / transversions if transversions > 0 else float('inf')
    
    return {
        'hamming': hamming,
        'transitions': transitions,
        'transversions': transversions,
        'ti_tv_ratio': ti_tv_ratio,
        'details': mutation_details
    }

# Test
s1 = "ATGCGATCGA"
s2 = "ATGCAATCTA"
result = mutation_analysis(s1, s2)
print(f"Seq 1: {s1}")
print(f"Seq 2: {s2}")
print(f"Hamming distance:   {result['hamming']}")
print(f"Transitions (Ti):   {result['transitions']}")
print(f"Transversions (Tv): {result['transversions']}")
print(f"Ti/Tv ratio:        {result['ti_tv_ratio']:.2f}")
print(f"\nMutation details:")
for pos, n1, n2, mut_type in result['details']:
    print(f"  Position {pos}: {n1}->{n2} ({mut_type})")

---

## Summary

### Key Control Flow Patterns for Bioinformatics

| Pattern | Use Case |
|---------|----------|
| `if/elif/else` | Classify nucleotides, amino acids, sequences |
| `for nuc in seq` | Iterate over each nucleotide |
| `for i in range(0, n, 3)` | Extract codons from a sequence |
| `for i, nuc in enumerate(seq)` | Track position during iteration |
| `for n1, n2 in zip(seq1, seq2)` | Compare aligned sequences |
| `while pos != -1` | Find all occurrences of a motif |
| `break` | Stop at first stop codon or invalid character |
| `continue` | Skip invalid or unwanted elements |
| `for ... else` | Check if a loop completed without breaking |
| Nested loops | Generate all codons, scan all reading frames |

### Key Biological Concepts Covered
- **Codon translation** using dictionary lookup
- **ORF finding** across multiple reading frames
- **Palindromic sequences** (restriction enzyme sites)
- **Sliding window analysis** for local sequence properties
- **Sequence validation** and classification
- **Mutation analysis** (transitions vs. transversions)

### Next Module: Lists, Tuples, and Dictionaries -->

---

[← Previous: Module 4: Strings and Biological Sequences](../../Tier_1_Python_for_Bioinformatics/04_Strings_and_Sequences/01_strings_and_sequences.ipynb) | [Next: Module 6: Functions in Python →](../../Tier_1_Python_for_Bioinformatics/06_Functions/01_functions.ipynb)